## Installation
Install the decent-bench package, PyTorch, and the Hugging Face datasets package if not already present.

In [ ]:
%pip install -e ../../.
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install datasets

## Imports

In [ ]:
from pathlib import Path

import torch
from src.dataset import FEMNISTDatasetHandler
from torch import nn

import decent_bench.algorithms.federated as dec_algorithms
import decent_bench.utils.interoperability as iop
from decent_bench import benchmark
from decent_bench.agents import Agent
from decent_bench.algorithms.utils import pytorch_initialization
from decent_bench.costs import PyTorchCost
from decent_bench.metrics import metric_library as ml
from decent_bench.networks import FedNetwork
from decent_bench.schemes import GaussianNoise, TopK, UniformActivationRate, UniformDropRate, UniformSelection
from decent_bench.utils.checkpoint_manager import CheckpointManager
from decent_bench.utils.pytorch_utils import ArgmaxActivation, SimpleLinearModel
from decent_bench.utils.types import SupportedDevices

## Experiment parameters

In [ ]:
checkpoint_path = Path("results/femnist")
iterations = 1000
n_trials = 3
state_snapshot_period = 150
progress_step = 150
n_clients = 10
min_train_samples = 50
min_test_samples = 10
max_samples_per_client = 100
train_fraction = 0.8
selection_fraction = 0.1
batch_size = 32
device = SupportedDevices.CPU

# Ideal communication is the default. Uncomment the lines below to test a non-ideal communication setting.
activation = None
noise = None
compression = None
drops = None

# activation = UniformActivationRate(0.8)
# noise = GaussianNoise(0.0, 0.01)
# compression = TopK(0.1)
# drops = UniformDropRate(0.2)

# Set seed
iop.set_seed(20260524)

## Define which metrics to calculate

In [ ]:
table_metrics = [
    ml.ServerAccuracy(fmt=".2%", x_log=False, y_log=False),
    ml.Accuracy(fmt=".2%", x_log=False, y_log=False),
    ml.Loss(),
    ml.ClientDriftFromServer(x_log=False, y_log=False),
    ml.GradientCalls(),
    ml.SentMessages(),
    ml.ReceivedMessages(),
]

plot_metrics = [
    ml.ServerAccuracy(fmt=".2%", x_log=False, y_log=False),
    ml.Accuracy(fmt=".2%", x_log=False, y_log=False),
    ml.Loss(),
    ml.ClientDriftFromServer(x_log=False, y_log=False),
]

## Define PyTorch model generator

In [ ]:
def model_generator() -> torch.nn.Module:
    """Generate a compact CPU-friendly model for flattened FEMNIST images."""
    return SimpleLinearModel(
        input_size=28 * 28,
        hidden_sizes=[64],
        output_size=62,
    )

## Create FEMNIST datasets
FEMNIST (Federated Extended MNIST) is a federated image classification dataset derived from the NIST Special
Database 19. It consists of handwritten characters (digits 0-9, uppercase A-Z, and lowercase a-z, giving 62
classes) written by many different writers. Each writer's samples form a natural non-IID partition, making
FEMNIST well-suited for federated learning research. Here we select a subset of writers, treat each writer
as a separate client, and cap the number of samples per client to keep training feasible on CPU.

FEMNIST is part of the LEAF benchmark (Caldas et al., 2018; https://github.com/TalwalkarLab/leaf).
Data is loaded from the `flwrlabs/femnist` Hugging Face dataset, which mirrors the LEAF partitioning.

In [ ]:
train_dataset = FEMNISTDatasetHandler(
    split="train",
    n_clients=n_clients,
    train_fraction=train_fraction,
    min_train_samples=min_train_samples,
    min_test_samples=min_test_samples,
    max_samples_per_client=max_samples_per_client,
)
test_dataset = FEMNISTDatasetHandler(
    split="test",
    n_clients=n_clients,
    train_fraction=train_fraction,
    min_train_samples=min_train_samples,
    min_test_samples=min_test_samples,
    max_samples_per_client=max_samples_per_client,
)
train_partitions = train_dataset.get_partitions()
test_data = test_dataset.get_datapoints()

## Create checkpoint manager

In [ ]:
cm = CheckpointManager(
    checkpoint_dir=checkpoint_path,
    checkpoint_step=None,
    keep_n_checkpoints=1,
    benchmark_metadata={
        "dataset": "FEMNIST",
        "dataset_source": "flwrlabs/femnist",
        "n_clients": n_clients,
        "clients_per_round": max(1, int(selection_fraction * n_clients)),
        "selection_fraction": selection_fraction,
        "max_samples_per_client": max_samples_per_client,
        "n_trials": n_trials,
        "iterations": iterations,
        "state_snapshot_period": state_snapshot_period,
        "checkpoint_step": None,
        "device": str(device),
        "drops": drops.__class__.__name__ if drops else None,
        "activity": activation.__class__.__name__ if activation else None,
        "compression": compression.__class__.__name__ if compression else None,
        "noise": noise.__class__.__name__ if noise else None,
    },
)

## Create costs, agents, network, and benchmark problem

In [ ]:
costs = [
    PyTorchCost(
        dataset=partition,
        model=model_generator(),
        loss_fn=nn.CrossEntropyLoss(),
        final_activation=ArgmaxActivation(),
        batch_size=min(batch_size, len(partition)),
        device=device,
        load_dataset=True,
    )
    for partition in train_partitions
]
agents = [
    Agent(
        cost,
        activation=activation,
        state_snapshot_period=state_snapshot_period,
    )
    for cost in costs
]
network = FedNetwork(
    clients=agents,
    message_noise=noise,
    message_compression=compression,
    message_drop=drops,
)
problem = benchmark.BenchmarkProblem(
    network=network,
    test_data=test_data,
)

## Create algorithms

In [ ]:
def make_selection_scheme() -> UniformSelection:
    """Create a fresh client selection scheme for each algorithm."""
    return UniformSelection(fraction_selected_clients=selection_fraction)


x0 = pytorch_initialization(network, all_same=True)
algorithms = [
    dec_algorithms.FedAvg(
        iterations=iterations,
        step_size=0.1,
        num_local_steps=4,
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
    dec_algorithms.FedProx(
        iterations=iterations,
        step_size=0.1,
        num_local_steps=4,
        penalty=0.025887619090591573,
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
    dec_algorithms.FedNova(
        iterations=iterations,
        step_size=0.015780201353739066,
        num_local_steps=3,
        use_momentum=True,
        momentum=0.5,
        use_server_momentum=True,
        server_momentum=0.9,
        use_prox=False,
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
    dec_algorithms.FedAdam(
        iterations=iterations,
        step_size=0.016454811464286817,
        num_local_steps=7,
        server_step_size=0.005781649782731609,
        beta_1=0.9,
        beta_2=0.9,
        epsilon=0.001,
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
    dec_algorithms.FedLT(
        iterations=iterations,
        step_size=0.0015,
        num_local_steps=5,
        penalty=1.0,
        local_solver="adam",
        solver_args={"beta1": 0.5, "beta2": 0.999, "epsilon": 1e-8},
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
    dec_algorithms.FedDyn(
        iterations=iterations,
        step_size=0.02760842017693185,
        num_local_steps=2,
        penalty=1.0,
        selection_scheme=make_selection_scheme(),
        x0=x0,
    ),
]

## Start benchmark execution

In [ ]:
result = benchmark.benchmark(
    algorithms=algorithms,
    benchmark_problem=problem,
    n_trials=n_trials,
    progress_step=progress_step,
    show_speed=True,
    show_trial=True,
    checkpoint_manager=cm,
)

metric_result = benchmark.compute_metrics(
    benchmark_result=result,
    checkpoint_manager=cm,
    table_metrics=table_metrics,
    plot_metrics=plot_metrics,
    statistics_across_agents=["mean", "min", "max"],
)

benchmark.display_metrics(
    metrics_result=metric_result,
    checkpoint_manager=cm,
    individual_plots=True,
)

## Optionally resume benchmark from checkpoint
Uncomment the code in the code block below to resume an interrupted benchmark execution

In [ ]:
# result = benchmark.resume_benchmark(
#     checkpoint_manager=cm,
#     create_backup=False,
#     show_speed=True,
#     show_trial=True,
# )

# metric_result = benchmark.compute_metrics(
#     benchmark_result=result,
#     checkpoint_manager=cm,
#     table_metrics=table_metrics,
#     plot_metrics=plot_metrics,
#     statistics_across_agents=["mean", "min", "max"],
# )

# benchmark.display_metrics(
#     metrics_result=metric_result,
#     checkpoint_manager=cm,
#     individual_plots=True,
# )